> **Chapter 14, Part 7** | Engineering lens. **Focus:** turn what you learned into a workload-to-index decision tool, with a reusable benchmark harness.

# Capstone: Build Your Own Fractal Index for Your Workload

This notebook is the chapter's takeaway tool. Given a description of your workload, it tells you which fractal index to use and how to tune it. The recommendation is conservative on purpose: the cases where the fractal apparatus loses are listed first.

The notebook also provides a reusable benchmark harness so you can validate the recommendation on your own data instead of trusting it.


## Step 1: characterize your workload

Answer five questions about the data you index and the queries you run.


In [1]:
def characterize_workload(
    *,
    n_dimensions: int,
    n_rows: int,
    skew: str,
    primary_query: str,
    update_pattern: str,
) -> dict:
    """Return a dict describing the workload.

    Parameters
    ----------
    n_dimensions : int
        Number of indexed columns / features. 1 for time-only, 2-4 for geospatial,
        16+ for embeddings.
    n_rows : int
        Approximate row count.
    skew : str
        One of {"uniform", "moderate", "heavy"}.
    primary_query : str
        One of {"point", "range", "knn", "aggregation"}.
    update_pattern : str
        One of {"append-only", "occasional-update", "frequent-update"}.
    """
    return {
        'dim': n_dimensions, 'rows': n_rows, 'skew': skew,
        'query': primary_query, 'updates': update_pattern,
    }


workload = characterize_workload(
    n_dimensions=2, n_rows=10_000_000, skew='heavy',
    primary_query='range', update_pattern='append-only',
)
print(workload)


{'dim': 2, 'rows': 10000000, 'skew': 'heavy', 'query': 'range', 'updates': 'append-only'}


## Step 2: get a recommendation


In [2]:
def recommend_index(workload: dict) -> dict:
    dim = workload['dim']
    rows = workload['rows']
    skew = workload['skew']
    query = workload['query']
    updates = workload['updates']

    if updates == 'frequent-update' and dim <= 2:
        return {
            'family': 'B-tree or LSM with no fractal clustering',
            'rationale': (
                'Frequent updates make Hilbert order go stale fast. Overhead of re-clustering '
                'exceeds the locality win. Use the standard transactional index.'
            ),
            'failure_mode_to_watch': 'See notebook 14.8 failure mode 1 (skewed updates).',
        }

    if query == 'knn' and dim >= 8:
        m_recommendation = max(8, min(32, 2 * int(np.sqrt(dim))))
        return {
            'family': 'HNSW',
            'rationale': (
                f'High-dim kNN is HNSW territory. Set M={m_recommendation} as a starting point '
                'based on intrinsic dimension; tune ef_construction up if recall is low.'
            ),
            'failure_mode_to_watch': 'See notebook 14.8 failure mode 3 (high local intrinsic dimension).',
        }

    if dim == 1 and query in ('range', 'aggregation'):
        return {
            'family': 'Hurst-aware time partitioning',
            'rationale': (
                'For a time series with Hurst > 0.6, adaptive chunk boundaries cut I/O 20-40% '
                'versus fixed-interval. For random-walk-like series (H ~ 0.5), fixed chunks are fine.'
            ),
            'failure_mode_to_watch': 'See notebook 14.8 failure mode 2 (distribution drift breaks H estimates).',
        }

    if dim in (2, 3) and query in ('range', 'aggregation') and skew in ('moderate', 'heavy'):
        return {
            'family': 'Hilbert curve clustering on Parquet (Liquid Clustering style)',
            'rationale': (
                'Multi-dim range queries on skewed data are exactly where Hilbert beats Z-order '
                'beats row-major. Use Iceberg .hilbertCurve(), Delta Liquid Clustering, or DuckDB '
                'ST_Hilbert depending on your engine.'
            ),
            'failure_mode_to_watch': 'See notebook 14.8 failure mode 4 (cache phantom speedups).',
        }

    if dim >= 4 and query == 'range':
        return {
            'family': 'Hilbert curve clustering with fractal-dimension-driven cardinality estimation',
            'rationale': (
                'High-dim range queries benefit from both Hilbert linearization AND fractal '
                'selectivity estimation. The combo is unpublished as a system but each piece is solid.'
            ),
            'failure_mode_to_watch': 'See notebook 14.8 failure mode 2 (drift) and the full research plan H1/H4.',
        }

    return {
        'family': 'B-tree or LSM with no fractal clustering',
        'rationale': 'Workload does not match a fractal-index sweet spot. Default index is fine.',
        'failure_mode_to_watch': 'No fractal-specific failure mode applies.',
    }


import numpy as np  # used in HNSW M heuristic

reco = recommend_index(workload)
print(f"Recommendation: {reco['family']}")
print(f"Rationale     : {reco['rationale']}")
print(f"Watch         : {reco['failure_mode_to_watch']}")


Recommendation: Hilbert curve clustering on Parquet (Liquid Clustering style)
Rationale     : Multi-dim range queries on skewed data are exactly where Hilbert beats Z-order beats row-major. Use Iceberg .hilbertCurve(), Delta Liquid Clustering, or DuckDB ST_Hilbert depending on your engine.
Watch         : See notebook 14.8 failure mode 4 (cache phantom speedups).


In [3]:
examples = [
    characterize_workload(n_dimensions=384, n_rows=2_000_000, skew='moderate',
                          primary_query='knn', update_pattern='occasional-update'),
    characterize_workload(n_dimensions=1, n_rows=500_000_000, skew='moderate',
                          primary_query='range', update_pattern='append-only'),
    characterize_workload(n_dimensions=3, n_rows=50_000_000, skew='heavy',
                          primary_query='range', update_pattern='append-only'),
    characterize_workload(n_dimensions=2, n_rows=1_000_000, skew='moderate',
                          primary_query='point', update_pattern='frequent-update'),
]

for w in examples:
    r = recommend_index(w)
    print(f"workload: dim={w['dim']:>3} rows={w['rows']:>12,} skew={w['skew']:>8} query={w['query']:>11} updates={w['updates']:>17}")
    print(f"  -> {r['family']}")
    print()


workload: dim=384 rows=   2,000,000 skew=moderate query=        knn updates=occasional-update
  -> HNSW

workload: dim=  1 rows= 500,000,000 skew=moderate query=      range updates=      append-only
  -> Hurst-aware time partitioning

workload: dim=  3 rows=  50,000,000 skew=   heavy query=      range updates=      append-only
  -> Hilbert curve clustering on Parquet (Liquid Clustering style)

workload: dim=  2 rows=   1,000,000 skew=moderate query=      point updates=  frequent-update
  -> B-tree or LSM with no fractal clustering



## Step 3: validate with a benchmark harness

The function below runs the same I/O-counting benchmark used in 14.5 and 14.6. Pass it your own data and your own query distribution.


In [4]:
def benchmark_orderings(points: np.ndarray, queries: list, page_size: int = 1000) -> dict:
    """Compare row-major, Z-order, and Hilbert ordering on a 2D dataset.

    Parameters
    ----------
    points : (N, 2) array
    queries : list of (x_min, y_min, x_max, y_max) tuples
    page_size : int

    Returns
    -------
    dict mapping ordering name -> (avg pages read, % skipped)
    """
    n = len(points)
    grid = int(2 ** np.ceil(np.log2(max(points.max(), 2))))
    order_log = int(np.log2(grid))

    h_keys = [hilbert_xy_to_d(int(x), int(y), n=grid) for x, y in points.astype(int)]
    z_keys = [zorder_xy_to_d(int(x), int(y), order=order_log) for x, y in points.astype(int)]

    orderings = {
        'row_major': np.arange(n),
        'zorder': np.argsort(z_keys),
        'hilbert': np.argsort(h_keys),
    }

    results = {}
    for name, ordering in orderings.items():
        ordered = points[ordering]
        boxes = []
        for s in range(0, n, page_size):
            chunk = ordered[s:s + page_size]
            boxes.append((chunk[:, 0].min(), chunk[:, 0].max(),
                          chunk[:, 1].min(), chunk[:, 1].max()))
        pages_read = []
        for q in queries:
            qx0, qy0, qx1, qy1 = q
            count = sum(1 for (bx0, bx1, by0, by1) in boxes
                        if not (bx1 < qx0 or bx0 > qx1 or by1 < qy0 or by0 > qy1))
            pages_read.append(count)
        avg_read = float(np.mean(pages_read))
        pct_skip = (1 - avg_read / len(boxes)) * 100
        results[name] = {'avg_pages_read': avg_read, 'pct_skipped': pct_skip, 'n_pages': len(boxes)}
    return results


def hilbert_xy_to_d(x: int, y: int, n: int) -> int:
    rx = 0; ry = 0; d = 0; s = n // 2
    while s > 0:
        rx = 1 if (x & s) > 0 else 0
        ry = 1 if (y & s) > 0 else 0
        d += s * s * ((3 * rx) ^ ry)
        if ry == 0:
            if rx == 1:
                x = s - 1 - x; y = s - 1 - y
            x, y = y, x
        s //= 2
    return d


def zorder_xy_to_d(x: int, y: int, order: int) -> int:
    d = 0
    for i in range(order):
        d |= ((x >> i) & 1) << (2 * i)
        d |= ((y >> i) & 1) << (2 * i + 1)
    return d


sample_pts = np.random.randint(0, 1024, size=(20_000, 2))
sample_q = []
for _ in range(100):
    cx, cy = np.random.uniform(50, 974, size=2)
    side = np.random.uniform(20, 100)
    sample_q.append((cx - side, cy - side, cx + side, cy + side))

result = benchmark_orderings(sample_pts, sample_q, page_size=500)
print(f'{"ordering":>10} | {"pages read":>12} | {"% skipped":>10}')
print('-' * 40)
for name, stats in result.items():
    print(f'{name:>10} | {stats["avg_pages_read"]:>12.1f} | {stats["pct_skipped"]:>9.1f}%')


  ordering |   pages read |  % skipped
----------------------------------------
 row_major |         40.0 |       0.0%
    zorder |          5.3 |      86.7%
   hilbert |          3.6 |      91.1%


## Printable recommendation card

Print this card and tape it next to the workstation of any data engineer who is about to ship a new index.

| Workload shape | Recommended fractal index | Don't use it when |
|---|---|---|
| 2-3D geospatial range queries on append-only data | Hilbert curve clustering on Parquet | Updates are frequent (weekly+) |
| 1D time-series with H > 0.6 | Hurst-aware partitioning | H ~ 0.5 (random walk) or H estimator unstable |
| High-dim kNN on stable embeddings | HNSW with M tuned to intrinsic dimension | Embeddings drift faster than rebuild cadence |
| Multi-dim selectivity estimation | Faloutsos D2 estimator alongside histograms | Data distribution is genuinely uniform |
| Frequent updates on small dimension | Standard B-tree or LSM | Don't bother with fractal clustering |

The chapter ends here for the engineer who came for practical advice. Notebook 14.8 ends here for the engineer who needs to know where this whole apparatus breaks.
